# Government QA Result Consolidation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
PATHS = {
    "M0": "/Volumes/main/default/thesis_project/M0/Test_1.1/gov_m0_answers_1.1.parquet",
    "M1": "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_M1_answers.2.2.parquet",
    "M2": "/Volumes/main/default/thesis_project/M2_NoContext/M2_Test/G_test_M2_1.7.parquet",
    "M3": "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/pred_govqa_test_M3_frozen_1.0.parquet",
}

In [ ]:
!pip -q install pyarrow >/dev/null

import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

def show_columns(path: str):
    p = Path(path)
    print("="*80)
    print(f"File: {p}")
    if not p.exists():
        print("❌ 文件不存在"); return
    try:
        if p.suffix.lower() == ".parquet":
            pf = pq.ParquetFile(str(p))
            names = [f.name for f in pf.schema]  # 列名
            nrows = pf.metadata.num_rows if pf.metadata is not None else "?"
            print(f"✅ 识别为 Parquet | 行数: {nrows}")
            print("Columns:", names)
        elif p.suffix.lower() in {".csv"}:
            df = pd.read_csv(p, nrows=0)
            print("✅ 识别为 CSV")
            print("Columns:", list(df.columns))
        elif p.suffix.lower() in {".jsonl", ".json"}:
            try:
                df = pd.read_json(p, lines=True, nrows=0)
            except ValueError:
                df = pd.read_json(p, nrows=0)
            print("✅ 识别为 JSON/JSONL")
            print("Columns:", list(df.columns))
        else:
            # 回退：尝试用 pandas 自动识别
            df = pd.read_parquet(p) if p.suffix.lower() in {".pqt", ".pq"} else pd.read_parquet(p)
            print("⚠️ 非标准后缀，尝试按 Parquet 读取")
            print("Columns:", list(df.columns))
    except Exception as e:
        print("❗ 读取失败：", e)


In [ ]:
for name, path in PATHS.items():
    print(f"\n>>> {name}")
    show_columns(path)

In [ ]:
# 预览前 3 行（所有列）
import pandas as pd
for name, path in PATHS.items():
    print(f"\n--- {name} head(3) ---")
    df = pd.read_parquet(path)  # 若是 csv/jsonl，换成 read_csv/read_json(lines=True)
    display(df.head(3))


In [ ]:
# ================== 0) 路径配置（请按实际情况修改） ==================
from pathlib import Path
import pandas as pd, numpy as np, json, ast, re

# 四个模型输出
P_M0 = Path("/Volumes/main/default/thesis_project/M0/Test_1.1/gov_m0_answers_1.1.parquet")
P_M1 = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_M1_answers.2.2.parquet")
P_M2 = Path("/Volumes/main/default/thesis_project/M2_NoContext/M2_Test/G_test_M2_1.7.parquet")
P_M3 = Path("/Volumes/main/default/thesis_project/M3/Govern_M3_Test/pred_govqa_test_M3_frozen_1.0.parquet")

# 候选 meta（M1/M3 的 chunk_id/pid -> 原文），按实际补充你的路径
META_CANDIDATES = [
    "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_meta.jsonl",
    "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/test_GovReport_e5_1.0/govqa_test_topk.jsonl",
]

OUT_UNIFIED = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_all_models_unified.1.0.parquet")

In [ ]:
# ================== 1) 工具函数 ==================
import hashlib
def _ensure_list_str(x):
    if isinstance(x, (list,tuple)):
        return [str(t) for t in x if isinstance(t,(str,bytes)) and str(t).strip()]
    # numpy array 兼容
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception:
        pass
    # 兼容字符串形式的 list
    if isinstance(x, str) and x.strip():
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                obj = json.loads(s);
                if isinstance(obj, list):
                    return [str(t) for t in obj if str(t).strip()]
            except Exception:
                try:
                    obj = ast.literal_eval(s)
                    if isinstance(obj,(list,tuple)):
                        return [str(t) for t in obj if str(t).strip()]
                except Exception:
                    pass
        # 常见分隔
        if "|||" in s:
            return [p.strip() for p in s.split("|||") if p.strip()]
        if "\n" in s:
            return [p.strip() for p in s.split("\n") if p.strip()]
    return []

def norm_q(s):
    return re.sub(r"\s+"," ", str(s or "").strip())

def stable_qid(question: str) -> str:
    return hashlib.sha1(norm_q(question).encode("utf-8")).hexdigest()

# 解析 M3/M1 里可能出现的 “段ID/pid/chunk_id”
PID_TOKEN_RE = re.compile(r"[A-Z]\d{4,}_(?:p|P)\d+")
def extract_pid_tokens(text: str):
    s = str(text or "")
    return PID_TOKEN_RE.findall(s)

# 读取 meta.jsonl -> 两类映射：pid->text 和 chunk_id->text（尽量兼容字段名）
def load_meta_maps(candidates):
    pid_map = {}
    chunk_map = {}
    import os, json
    for path in candidates:
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line=line.strip()
                if not line: continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                txt = obj.get("text") or obj.get("content") or obj.get("paragraph") or obj.get("raw") or ""
                if not isinstance(txt,str) or not txt.strip():
                    continue
                # 常见键名尝试
                for k in ["pid","paragraph_id","para_id"]:
                    if k in obj and str(obj[k]).strip():
                        pid_map[str(obj[k]).strip()] = txt
                # chunk_id 形态
                for k in ["chunk_id","chunkId","cid","id"]:
                    if k in obj and str(obj[k]).strip():
                        chunk_map[str(obj[k]).strip()] = txt
                # 组合形态（doc_id + pid）
                if "doc_id" in obj and "pid" in obj:
                    pid_map[f"{obj['doc_id']}_{obj['pid']}"] = txt
    return pid_map, chunk_map

pid_map, chunk_map = load_meta_maps(META_CANDIDATES)
print(f"[META] pid_map={len(pid_map)} | chunk_map={len(chunk_map)}")

In [ ]:
# ================== 2) 读取四个模型 ==================
M0 = pd.read_parquet(P_M0)
M1 = pd.read_parquet(P_M1)
M2 = pd.read_parquet(P_M2)
M3 = pd.read_parquet(P_M3)

print("M0 cols:", list(M0.columns))
print("M1 cols:", list(M1.columns))
print("M2 cols:", list(M2.columns))
print("M3 cols:", list(M3.columns))

In [ ]:
# ================== 3) 为 M3 构造 retrieved_ctx ==================
# 优先使用 M3['contexts']（如存在且为文本列表）；否则用 pid_list + meta 映射；再否则尝试从 topk_sha256 不做处理（留空）。
def build_m3_retrieved_ctx(df: pd.DataFrame) -> pd.Series:
    # contexts 列（若已是文本列表）
    if "contexts" in df.columns:
        ctx = df["contexts"].apply(_ensure_list_str)
        # contexts 有些是带前缀 "[s2] ..." 可接受；保持原文
        non_empty = int(ctx.apply(lambda x: len(x)>0).sum())
        print(f"[M3] contexts 列可用：非空 {non_empty}/{len(df)}")
        if non_empty > 0:
            return ctx

    # pid_list + meta
    if "pid_list" in df.columns and len(pid_map)>0:
        def map_pids(pids):
            out=[]
            for p in _ensure_list_str(pids):
                # 精确匹配
                if p in pid_map:
                    out.append(pid_map[p]); continue
                # 正则提取（如整段字符串）
                for token in extract_pid_tokens(p):
                    if token in pid_map:
                        out.append(pid_map[token])
            return out
        mapped = df["pid_list"].apply(map_pids)
        non_empty = int(mapped.apply(lambda x: len(x)>0).sum())
        print(f"[M3] 通过 pid_list+meta 映射：非空 {non_empty}/{len(df)}")
        return mapped

    # 兜底：空列表
    print("[M3] 未找到可用的 contexts/pid_list 或 meta 映射，retrieved_ctx 留空。")
    return pd.Series([[] for _ in range(len(df))], index=df.index)

M3_ctx = build_m3_retrieved_ctx(M3)


In [ ]:
# ================== 4) 为 M1 构造 retrieved_ctx 和 oracle_ctx ==================
# oracle_ctx：来自 oracle_evidence（单字符串 -> [str]）
def build_m1_oracle_ctx(df: pd.DataFrame) -> pd.Series:
    if "oracle_evidence" in df.columns:
        oc = df["oracle_evidence"].apply(lambda s: [str(s)] if isinstance(s,str) and s.strip() else [])
        print(f"[M1] oracle_evidence 非空：{int(oc.apply(len).gt(0).sum())}/{len(df)}")
        return oc
    print("[M1] 无 oracle_evidence，oracle_ctx 留空。")
    return pd.Series([[] for _ in range(len(df))], index=df.index)

# retrieved_ctx：优先使用显式列（retrieved_chunks / contexts / pid_list），再尝试从 path/source_file / report_id+summary_para_id 推断 PID，最后用 meta 映射到文本
def build_m1_retrieved_ctx(df: pd.DataFrame) -> pd.Series:
    # 1) 直接文本列
    for cname in ["contexts"]:
        if cname in df.columns:
            cand = df[cname].apply(_ensure_list_str)
            if int(cand.apply(lambda x: len(x)>0).sum())>0:
                print(f"[M1] 使用文本列 {cname} 作为 retrieved_ctx")
                return cand

    # 2) chunk/pid 列 + meta
    id_cols = [c for c in ["retrieved_chunks","pid_list","topk_sha256","path"] if c in df.columns]
    if id_cols and (len(pid_map)>0 or len(chunk_map)>0):
        def ids_to_text(x):
            ids = _ensure_list_str(x)
            out=[]
            for token in ids:
                # 直接命中
                if token in pid_map: out.append(pid_map[token]); continue
                if token in chunk_map: out.append(chunk_map[token]); continue
                # 从长串里提取 PID
                for pid in extract_pid_tokens(token):
                    if pid in pid_map: out.append(pid_map[pid])
            return out
        # 逐列尝试，优先 retrieved_chunks / pid_list / path
        for cname in ["retrieved_chunks","pid_list","path","topk_sha256"]:
            if cname in df.columns:
                mapped = df[cname].apply(ids_to_text)
                non_empty = int(mapped.apply(lambda x: len(x)>0).sum())
                print(f"[M1] 通过 {cname}+meta 映射：非空 {non_empty}/{len(df)}")
                if non_empty>0:
                    return mapped

    # 3) report_id + summary_para_id -> 组合 pid
    if "report_id" in df.columns and "summary_para_id" in df.columns and len(pid_map)>0:
        def pair_to_text(row):
            rid = str(row.get("report_id","")).strip()
            sp  = str(row.get("summary_para_id","")).strip()
            cands = []
            if rid and sp:
                # 常见 pid 形态：R12345_p678
                for pid in [f"{rid}_{sp}", f"{rid}_p{sp}", f"{rid}_P{sp}"]:
                    if pid in pid_map:
                        cands.append(pid_map[pid])
            return cands
        mapped = df.apply(pair_to_text, axis=1)
        non_empty = int(mapped.apply(lambda x: len(x)>0).sum())
        print(f"[M1] report_id+summary_para_id 映射：非空 {non_empty}/{len(df)}")
        if non_empty>0:
            return mapped

    print("[M1] 未找到可用检索来源，retrieved_ctx 留空。")
    return pd.Series([[] for _ in range(len(df))], index=df.index)

M1_oracle = build_m1_oracle_ctx(M1)
M1_ctx    = build_m1_retrieved_ctx(M1)


In [ ]:
# ================== 5) 规范化为统一 schema ==================
def to_unified_M0(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "question": df["question"].astype(str),
        "answer":   df["answer_M0"].astype(str),
        "gold":     np.nan,
        "paper_id": np.nan,
        "oracle_ctx": [[] for _ in range(len(df))],
        "retrieved_ctx": [[] for _ in range(len(df))],
        "citations": [[] for _ in range(len(df))]
    })
    out["model"] = "M0"
    out["question_id"] = [stable_qid(q) for q in out["question"]]
    out["question_norm"] = out["question"].map(norm_q)
    return out

def to_unified_M1(df: pd.DataFrame, oc: pd.Series, rc: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame({
        "question": df["question"].astype(str),
        "answer":   df["answer_M1"].astype(str),
        "gold":     df["gold_answer"].astype(str) if "gold_answer" in df.columns else np.nan,
        "paper_id": df["report_id"].astype(str) if "report_id" in df.columns else (df["source_file"].astype(str) if "source_file" in df.columns else np.nan),
        "oracle_ctx": oc,
        "retrieved_ctx": rc,
        "citations": [[] for _ in range(len(df))]
    })
    out["model"] = "M1"
    out["question_id"] = df["qid"].astype(str) if "qid" in df.columns else [stable_qid(q) for q in out["question"]]
    out["question_norm"] = out["question"].map(norm_q)
    return out

def to_unified_M2(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "question": df["question"].astype(str),
        "answer":   df["answer_M2_1"].astype(str),
        "gold":     df["gold_answer"].astype(str) if "gold_answer" in df.columns else np.nan,
        "paper_id": np.nan,
        "oracle_ctx": [[] for _ in range(len(df))],
        "retrieved_ctx": [[] for _ in range(len(df))],
        "citations": [[] for _ in range(len(df))]
    })
    out["model"] = "M2"
    out["question_id"] = [stable_qid(q) for q in out["question"]]
    out["question_norm"] = out["question"].map(norm_q)
    return out

def to_unified_M3(df: pd.DataFrame, rc: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame({
        "question": df["question"].astype(str),
        "answer":   df["answer_M3"].astype(str),
        "gold":     df["gold_answer"].astype(str) if "gold_answer" in df.columns else np.nan,
        "paper_id": df["doc_id"].astype(str) if "doc_id" in df.columns else np.nan,
        "oracle_ctx": [[] for _ in range(len(df))],
        "retrieved_ctx": rc,
        "citations": [[] for _ in range(len(df))]
    })
    out["model"] = "M3"
    out["question_id"] = df["qid"].astype(str) if "qid" in df.columns else [stable_qid(q) for q in out["question"]]
    out["question_norm"] = out["question"].map(norm_q)
    return out

U0 = to_unified_M0(M0)
U1 = to_unified_M1(M1, M1_oracle, M1_ctx)
U2 = to_unified_M2(M2)
U3 = to_unified_M3(M3, M3_ctx)

UNIFIED = pd.concat([U0,U1,U2,U3], axis=0, ignore_index=True)
print("合并完成：", UNIFIED.shape)
print("各模型计数：\n", UNIFIED["model"].value_counts())
print("retrieved_ctx 非空计数（按模型）：\n", UNIFIED.assign(nonempty=UNIFIED["retrieved_ctx"].apply(lambda x: int(len(x)>0))).groupby("model")["nonempty"].sum())

In [ ]:
# ================== 6) 保存 & 快速检查 ==================
OUT_UNIFIED.parent.mkdir(parents=True, exist_ok=True)
UNIFIED.to_parquet(OUT_UNIFIED, index=False)
print("💾 写入：", OUT_UNIFIED)

# 采样看一眼映射是否成功
mask_ctx = UNIFIED["retrieved_ctx"].apply(lambda x: len(x)>0)
print("有检索上下文的样本数：", int(mask_ctx.sum()))
if int(mask_ctx.sum())>0:
    sample_row = UNIFIED[mask_ctx].iloc[0]
    print("\n—— 示例（含 retrieved_ctx 的一行）——")
    print("model:", sample_row["model"])
    print("paper_id:", sample_row["paper_id"])
    print("question:", sample_row["question"][:160])
    print("answer:", sample_row["answer"][:160])
    print("retrieved_ctx 段数:", len(sample_row["retrieved_ctx"]))
    print("retrieved_ctx[0] 预览:", (sample_row["retrieved_ctx"][0] or "")[:240])


In [ ]:
import pandas as pd, numpy as np, json, ast, re
from pathlib import Path

INP = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_all_models_unified.1.0.parquet")
OUT = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_all_models_unified.1.1.parquet")

DF = pd.read_parquet(INP).copy()
print(f"Loaded: {INP} -> rows={len(DF)}, cols={len(DF.columns)}")

# --- 保障 question_norm 存在且规范 ---
def norm_q(s):
    return re.sub(r"\s+", " ", str(s or "").strip())

if "question_norm" not in DF.columns:
    DF["question_norm"] = DF["question"].map(norm_q)
else:
    DF["question_norm"] = DF["question_norm"].map(norm_q)

# --- 保障 oracle_ctx 是 list[str] ---
def _ensure_list_str(x):
    if isinstance(x, (list, tuple)):
        return [str(t) for t in x if isinstance(t,(str,bytes)) and str(t).strip()]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception:
        pass
    if isinstance(x, str) and x.strip():
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            # JSON / Python 列表字符串
            try:
                obj = json.loads(s)
                if isinstance(obj, list):
                    return [str(t) for t in obj if str(t).strip()]
            except Exception:
                try:
                    obj = ast.literal_eval(s)
                    if isinstance(obj, (list, tuple)):
                        return [str(t) for t in obj if str(t).strip()]
                except Exception:
                    pass
        # 普通文本，按“整段”处理为单元素列表
        return [s]
    return []

DF["oracle_ctx"] = DF.get("oracle_ctx", [[]]*len(DF)).apply(_ensure_list_str)

# --- 以 M1 为权威来源，构建映射（question_norm -> gold / oracle_ctx）---
M1 = DF[DF["model"]=="M1"].copy()

# gold：取第一条非空 gold（按 question_norm 分组）
m1_gold_map = (
    M1.assign(_gold=M1["gold"].astype(str).str.strip())
      .replace({"_gold": {"nan": ""}})
      .groupby("question_norm")["_gold"]
      .agg(lambda s: next((x for x in s if x), ""))  # 取首个非空
      .to_dict()
)

# oracle_ctx：把同题的多条 oracle_ctx 合并去重
def _merge_ctx(series):
    seen=set(); out=[]
    for lst in series:
        for t in _ensure_list_str(lst):
            if t not in seen:
                seen.add(t); out.append(t)
    return out
m1_oracle_map = (
    M1.groupby("question_norm")["oracle_ctx"].agg(_merge_ctx).to_dict()
)

print(f"M1 map sizes -> gold: {len(m1_gold_map)} | oracle_ctx: {len(m1_oracle_map)}")

# --- 回填前统计 ---
def _non_empty_gold(x):
    return int(isinstance(x,str) and x.strip()!="")
def _non_empty_ctx(x):
    return int(isinstance(x,(list,tuple)) and len(x)>0)

print("\n[Before] non-empty counts by model")
print(DF.assign(
    has_gold = DF["gold"].apply(_non_empty_gold),
    has_orcl = DF["oracle_ctx"].apply(_non_empty_ctx)
).groupby("model")[["has_gold","has_orcl"]].sum())

# --- 执行回填（同题强制对齐到 M1，未命中保留原值）---
def fill_gold(row):
    g = m1_gold_map.get(row["question_norm"], "")
    return g if g else (row["gold"] if isinstance(row["gold"], str) else "")

def fill_oracle(row):
    ctx = m1_oracle_map.get(row["question_norm"], [])
    return ctx if (isinstance(ctx, list) and len(ctx)>0) else row["oracle_ctx"]

DF["gold_filled"]       = DF.apply(fill_gold, axis=1)
DF["oracle_ctx_filled"] = DF.apply(fill_oracle, axis=1)

# 冲突诊断：原 gold 与 M1 gold 不同的条数（只统计原 gold 非空的行）
mask_gold_nonempty = DF["gold"].apply(_non_empty_gold).astype(bool)
gold_conflicts = int((DF.loc[mask_gold_nonempty, "gold"].str.strip() != DF.loc[mask_gold_nonempty, "gold_filled"].str.strip()).sum())
print(f"\nGold conflicts (orig != filled) on originally non-empty rows: {gold_conflicts}")

# 覆盖（正式替换）
DF["gold"] = DF["gold_filled"]; DF.drop(columns=["gold_filled"], inplace=True)
DF["oracle_ctx"] = DF["oracle_ctx_filled"]; DF.drop(columns=["oracle_ctx_filled"], inplace=True)

# --- 回填后统计 ---
print("\n[After] non-empty counts by model")
print(DF.assign(
    has_gold = DF["gold"].apply(_non_empty_gold),
    has_orcl = DF["oracle_ctx"].apply(_non_empty_ctx)
).groupby("model")[["has_gold","has_orcl"]].sum())

# --- 保存 ---
OUT.parent.mkdir(parents=True, exist_ok=True)
DF.to_parquet(OUT, index=False)
print("\n💾 Saved:", OUT)

# 小样例看一眼
print("\nSamples (M0/M2/M3 回填后的几行)：")
for m in ["M0","M2","M3"]:
    ex = DF[DF["model"]==m].head(2)[["model","question","gold","oracle_ctx"]]
    display(ex)


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json, ast, re

P = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_all_models_unified.1.1.parquet")
DF = pd.read_parquet(P).copy()
print(f"Loaded: rows={len(DF)}, cols={len(DF.columns)}")

REQUIRED = ["question_id","model","question","answer","gold","paper_id",
            "oracle_ctx","retrieved_ctx","citations","question_norm"]
missing = [c for c in REQUIRED if c not in DF.columns]
print("Missing required columns:", missing)

# 将 list 型列规范为 list[str]
def _ensure_list_str(x):
    if isinstance(x, (list, tuple)):
        return [str(t) for t in x if str(t).strip()]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception: pass
    if isinstance(x, str) and x.strip():
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                obj = json.loads(s);
                if isinstance(obj, list): return [str(t) for t in obj if str(t).strip()]
            except Exception:
                try:
                    obj = ast.literal_eval(s)
                    if isinstance(obj, (list, tuple)): return [str(t) for t in obj if str(t).strip()]
                except Exception: pass
        return [s]  # 单字符串 → 单段
    return []

for c in ["oracle_ctx","retrieved_ctx","citations"]:
    if c in DF.columns:
        DF[c] = DF[c].apply(_ensure_list_str)

# 规范 question_norm
def norm_q(s): return re.sub(r"\s+"," ", str(s or "").strip())
DF["question_norm"] = DF["question_norm"].map(norm_q) if "question_norm" in DF.columns else DF["question"].map(norm_q)

print("Column dtypes (head):")
print(DF.dtypes.astype(str).sort_index().to_string()[:1000])


In [ ]:
print("\nCounts by model:")
print(DF["model"].value_counts())

print("\nNon-empty by model:")
def _nn_gold(x): return int(isinstance(x,str) and x.strip()!="")
def _nn_ctx(x):  return int(isinstance(x,(list,tuple)) and len(x)>0)

summary = DF.assign(
    has_gold = DF["gold"].apply(_nn_gold),
    has_orcl = DF["oracle_ctx"].apply(_nn_ctx),
    has_ctx  = DF["retrieved_ctx"].apply(_nn_ctx),
).groupby("model")[["has_gold","has_orcl","has_ctx"]].sum()
display(summary)

print("\nExpectations (Govern 与 Qasper 对齐):")
print("- gold: M0 可能为空；M1/M2/M3 应大多非空")
print("- oracle_ctx: 你要求按 M1 回填 → 各模型应一致且多数非空")
print("- retrieved_ctx: M1/M3 应大多非空，M0/M2 通常为空")
